# Day 19 · 跨程序的握手：認識 A2A Protocol

> 第三部・戰術編排　|　🔀 真實跨程序：notebook 會起兩個 A2A server 子程序，最後自動關閉

**前置需求**：🔑 需要 Gemini API 金鑰（第 9 節之後才用到）　📦 需要 `google-adk[a2a]`

**對應文章**：`Day 19 - 跨程序的握手：認識 A2A Protocol.md`

## 今天要學會

1. 判斷一個多 agent 需求該用 A2A 還是 local sub-agent，並**用實測延遲**說明代價
2. 看懂 A2A 在線上真正傳的東西：Agent Card、JSON-RPC、Task、SSE 串流
3. 親手走一遍 Task 生命週期：`INPUT_REQUIRED` 補件、長任務串流、中途取消
4. 知道 ADK 的 `RemoteA2aAgent` 把 A2A 的每一種回應**翻譯成哪一種 ADK Event**
5. 讓一支 agent **同時** consuming 與 exposing，串起三個程序

> ⚠️ 今天不是 mock。你會啟動兩個真的 HTTP 服務：
>
> | 程序 | 埠 | 誰寫的 | 用什麼寫 |
> |---|---|---|---|
> | `product_catalog_server.py` | 8941 | 「另一個團隊」 | 純 `a2a-sdk`，**沒有 ADK、沒有 LLM** |
> | `customer_service_server.py` | 8942 | 你 | ADK `LlmAgent` + `to_a2a()` |
>
> 兩個檔案都在 `servers/`，也可以另開終端機自己跑。**請從頭依序執行到最後一格。**

## 環境設定

每一天都是獨立的，這段設定刻意重複，讓你可以從任何一天開始。

In [1]:
import warnings

warnings.filterwarnings("ignore")

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from shared import ask, get_model, new_session, quiet

quiet()

import google.adk

print("google-adk", google.adk.__version__)

try:
    import a2a.types as T
    from importlib.metadata import version

    print("a2a-sdk  ", version("a2a-sdk"))
except ModuleNotFoundError:
    raise ModuleNotFoundError(
        "找不到 a2a 套件。兩個常見原因：\n"
        "  1. 沒裝 extra → uv sync（pyproject 已含 google-adk[a2a]）\n"
        "  2. notebook 選錯 kernel → 右上角改選本專案的 .venv\n"
        f"目前的 Python：{sys.executable}"
    ) from None

google-adk 2.8.0
a2a-sdk   1.1.2


## 1. 先問一句：這件事真的需要跨網路嗎？

到 Day 18 為止，你的 agent 團隊全部活在**同一個 Python 程序**裡：
`sub_agents`、`AgentTool`、`mode` 本質上都是記憶體內的函式呼叫。
A2A 要解決的是另一件事——**合作對象是一個獨立的服務**：跑在別台機器、別的團隊維護、甚至不是 Python。

| 該用 A2A | 不該用 A2A（改用 local sub-agent） |
|---|---|
| 對方是**獨立、standalone 的服務** | **內部程式碼組織**（例如 `DataValidator`） |
| 由**不同團隊或組織**維護 | **效能敏感**、高頻低延遲的內部操作 |
| 要接**不同語言或不同框架**的 agent | 需要**共享記憶體 / session state** |
| 要在元件之間強制**正式契約** | **簡單的 helper 函式** |

今天用文章裡的案例把它做成真的：**客服 agent 要查產品目錄，而產品目錄是另一個團隊的服務**。
為了證明「跨框架」不是口號，產品目錄刻意**不用 ADK** 寫。

```mermaid
flowchart LR
    subgraph NB["這個 notebook（程序 1）"]
        C1["httpx 手刻 JSON-RPC<br/>（第 3–8 節）"]
        C2["RemoteA2aAgent<br/>（第 9–10 節）"]
    end
    subgraph CS["customer_service_server（程序 2, :8942）"]
        A["ADK LlmAgent"] --> R["RemoteA2aAgent"]
    end
    subgraph PC["product_catalog_server（程序 3, :8941）"]
        E["a2a-sdk AgentExecutor<br/>（沒有 ADK）"]
    end
    C1 -- HTTP --> E
    C2 -- HTTP --> E
    NB -- "HTTP（第 11 節）" --> A
    R -- HTTP --> E
```

第 12 節最後會回來**量延遲**，用數字回答「為什麼不要把 `DataValidator` 拆成 A2A 服務」。

## 2. 啟動「另一個團隊」的 A2A 服務

先看 server 的核心。A2A server 在 `a2a-sdk` 裡只有三塊積木：

| 積木 | 做什麼 |
|---|---|
| `AgentExecutor.execute()` | 你的業務邏輯。**不回傳值**，而是把狀態變化丟進 `event_queue` |
| `DefaultRequestHandler` + `TaskStore` | 協定本身：收請求、存 Task、把事件轉成回應 |
| `create_agent_card_routes` / `create_jsonrpc_routes` | 兩個 HTTP 端點：`GET /.well-known/agent-card.json`、`POST /` |

ADK 的 `to_a2a()` 做的事，就是幫你把 ADK agent 包成一個 `AgentExecutor`，再組好這三塊。

In [2]:
import inspect

SERVERS = Path.cwd() / "servers"
sys.path.insert(0, str(SERVERS))
import product_catalog_server as pcs

print(inspect.getsource(pcs.CatalogExecutor))

class CatalogExecutor(AgentExecutor):
    """A2A server 的核心：收到請求 → 把狀態變化丟進 event_queue。"""

    async def execute(self, context: RequestContext, event_queue: EventQueue) -> None:
        task = context.current_task
        if task is None:
            # 第一次收到這個 task：先把 Task 物件本身發出去（狀態 SUBMITTED）
            task = new_task_from_user_message(context.message)
            await event_queue.enqueue_event(task)
        updater = TaskUpdater(event_queue, task.id, task.context_id)

        def say(text: str):
            return new_text_message(text, task_id=task.id, context_id=task.context_id)

        text = read_request_text(context)
        await updater.start_work(say("收到，查詢目錄中…"))

        if "盤點" in text:
            for i in range(1, 11):
                await asyncio.sleep(0.5)
                await updater.update_status(
                    TaskState.TASK_STATE_WORKING, say(f"盤點進度 {i * 10}%")
                )
            await updater.complete(say(f"盤點完成，共 {len(CATALOG)} 項商品"))
   

注意 `INPUT_REQUIRED` 那一段：**不是丟例外，也不是回 400**，
而是把狀態設成「需要補件」然後 `return`。呼叫方帶著同一個 `task_id` 回來時，框架會再呼叫一次 `execute()`，
這時 `context.current_task` 就不是 `None` 了。

現在用子程序把它跑起來。

In [3]:
import atexit
import shutil
import socket
import subprocess
import time

import httpx

HOST = "localhost"  # ⚠️ 全程用同一個字串，localhost ≠ 127.0.0.1（Day 20 會重現這個坑）
CATALOG_PORT = 8941
CS_PORT = 8942
WORK = Path.cwd() / "_day19"
shutil.rmtree(WORK, ignore_errors=True)
WORK.mkdir()

PROCS: dict[str, subprocess.Popen] = {}


def port_in_use(port: int) -> bool:
    with socket.socket() as s:
        return s.connect_ex((HOST, port)) == 0


def start_server(name: str, script: str, port: int, *args: str, timeout: float = 60) -> str:
    """起一個 server 子程序，等到 agent card 抓得到才回傳 base URL。"""
    if port_in_use(port):
        raise RuntimeError(f"埠 {port} 已被佔用——多半是上次沒跑到最後一格。先關掉那個程序再重跑。")
    log = WORK / f"{name}.log"
    proc = subprocess.Popen(
        [sys.executable, str(SERVERS / script), "--host", HOST, "--port", str(port), *args],
        stdout=log.open("w"), stderr=subprocess.STDOUT,
    )
    PROCS[name] = proc
    base = f"http://{HOST}:{port}"
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        if proc.poll() is not None:
            raise RuntimeError(f"{name} 啟動失敗：\n{log.read_text()[-1500:]}")
        try:
            if httpx.get(f"{base}/.well-known/agent-card.json", timeout=1).status_code == 200:
                print(f"✅ {name} 已在 {base} 上線（pid={proc.pid}）")
                return base
        except httpx.TransportError:
            time.sleep(0.3)
    raise TimeoutError(f"{name} 在 {timeout} 秒內沒起來，看 {log}")


def stop_all():
    for name, proc in PROCS.items():
        if proc.poll() is None:
            proc.terminate()
            try:
                proc.wait(timeout=10)
            except subprocess.TimeoutExpired:
                proc.kill()


atexit.register(stop_all)  # kernel 關掉時也會清理

CATALOG = start_server("product_catalog", "product_catalog_server.py", CATALOG_PORT)

✅ product_catalog 已在 http://localhost:8941 上線（pid=389191）


## 3. Discovery：Agent Card

A2A 的核心是一份 JSON——**Agent Card**，掛在固定路徑 `/.well-known/agent-card.json`。
呼叫方**第一步一定是先抓卡**：抓到之後才知道該往哪個網址、用哪種傳輸協定、對方會什麼。

這是一個普通的 HTTP GET，你用瀏覽器打開也看得到。

In [4]:
import json

CARD_URL = f"{CATALOG}/.well-known/agent-card.json"
raw_card = httpx.get(CARD_URL).json()
print(json.dumps(raw_card, ensure_ascii=False, indent=2))

{
  "name": "product_catalog_agent",
  "description": "產品目錄服務：依商品編號（SKU）查詢品名、價格與庫存，也能做全品項盤點。",
  "supportedInterfaces": [
    {
      "url": "http://localhost:8941/",
      "protocolBinding": "JSONRPC",
      "protocolVersion": "1.0"
    }
  ],
  "version": "1.0.0",
  "capabilities": {
    "streaming": true,
    "pushNotifications": false
  },
  "defaultInputModes": [
    "text/plain"
  ],
  "defaultOutputModes": [
    "text/plain",
    "application/json"
  ],
  "skills": [
    {
      "id": "lookup_product",
      "name": "查詢產品",
      "description": "給一個 SKU（格式如 A-100），回傳品名、價格、庫存。缺 SKU 時會要求補件。",
      "tags": [
        "catalog",
        "inventory"
      ],
      "examples": [
        "A-100 多少錢？",
        "B-200 還有貨嗎？"
      ]
    },
    {
      "id": "stocktake",
      "name": "全品項盤點",
      "description": "長時間任務，會持續回報進度，可取消。",
      "tags": [
        "catalog",
        "long-running"
      ],
      "examples": [
        "幫我盤點"
      ]
    }
  ]
}


逐欄對照文章的說明：

| 欄位 | 呼叫方拿來做什麼 |
|---|---|
| `supportedInterfaces[].url` | **真正要 POST 的位址**（不一定等於抓卡的位址） |
| `supportedInterfaces[].protocolBinding` | 要講哪種傳輸：`JSONRPC` / `GRPC` / `HTTP+JSON` |
| `supportedInterfaces[].protocolVersion` | 協定版本，決定請求要帶什麼 header（第 8 節會踩到） |
| `capabilities.streaming` | 能不能用 `SendStreamingMessage` 邊做邊回報 |
| `skills[]` | **給機器讀的路由表**。LLM 讀 `description` / `examples` 決定要不要把工作丟過來 |
| `defaultInputModes` / `defaultOutputModes` | 吃什麼、吐什麼（MIME type） |

這份 JSON 不是隨便定的，它是 protobuf `AgentCard` 的 JSON 形式——所以能被**嚴格解析**。
欄位打錯字，解析就會失敗，這就是文章說的「正式契約」。

In [5]:
from a2a.helpers.agent_card import display_agent_card
from google.protobuf.json_format import ParseDict, ParseError

card = ParseDict(raw_card, T.AgentCard())
display_agent_card(card)

print("\n=== 如果對方卡片欄位拼錯 ===")
try:
    ParseDict({**raw_card, "skils": []}, T.AgentCard())
except ParseError as e:
    print("❌", str(e)[:120])

                     AgentCard                      
--- General ---
Name        : product_catalog_agent
Description : 產品目錄服務：依商品編號（SKU）查詢品名、價格與庫存，也能做全品項盤點。
Version     : 1.0.0

--- Interfaces ---
  [0] http://localhost:8941/  (JSONRPC 1.0)

--- Capabilities ---
Streaming           : True
Push notifications  : False
Extended agent card : False

--- I/O Modes ---
Input  : text/plain
Output : text/plain, application/json

--- Skills ---
----------------------------------------------------
  ID          : lookup_product
  Name        : 查詢產品
  Description : 給一個 SKU（格式如 A-100），回傳品名、價格、庫存。缺 SKU 時會要求補件。
  Tags        : catalog, inventory
  Example     : A-100 多少錢？
  Example     : B-200 還有貨嗎？
----------------------------------------------------
  ID          : stocktake
  Name        : 全品項盤點
  Description : 長時間任務，會持續回報進度，可取消。
  Tags        : catalog, long-running
  Example     : 幫我盤點

=== 如果對方卡片欄位拼錯 ===
❌ Message type "lf.a2a.v1.AgentCard" has no field named "skils" at "AgentCard".
 Available 

## 4. 協定在線上長什麼樣：手刻一次 JSON-RPC

先不靠任何 SDK，直接用 `httpx` 發請求。A2A 的 JSON-RPC binding 規則很少：

- **一律 `POST` 到卡片上的 `url`**，body 是 JSON-RPC 2.0
- `method` 是 `SendMessage`、`SendStreamingMessage`、`GetTask`、`CancelTask`…（跟 gRPC 方法同名）
- ⚠️ header 要帶 **`A2A-Version: 1.0`**（為什麼，第 8 節揭曉）

`message` 的結構：`messageId`（呼叫方產生）、`role`、`parts`。`part` 可以是 `text`、`data`（結構化 JSON）、`url`、`raw`（二進位）。

In [6]:
import uuid

RPC_URL = card.supported_interfaces[0].url  # ← 從卡片讀，不要自己拼
HEADERS = {"A2A-Version": "1.0"}


def rpc(method: str, params: dict, headers: dict | None = HEADERS) -> dict:
    body = {"jsonrpc": "2.0", "id": uuid.uuid4().hex[:6], "method": method, "params": params}
    return httpx.post(RPC_URL, json=body, headers=headers, timeout=30).json()


def user_msg(text: str, **ids) -> dict:
    """ids 可以帶 taskId / contextId，用來續接既有的 task 或對話。"""
    return {"message": {"messageId": uuid.uuid4().hex, "role": "ROLE_USER",
                        "parts": [{"text": text}], **ids}}


resp = rpc("SendMessage", user_msg("A-100 多少錢？"))
print(json.dumps(resp, ensure_ascii=False, indent=2))

{
  "result": {
    "task": {
      "id": "a4b57a5b-2340-43b4-a1fd-7e4b36327781",
      "contextId": "5bd6d5ec-31f4-48ad-aa1f-02e7f0ab6c3f",
      "status": {
        "state": "TASK_STATE_COMPLETED",
        "message": {
          "messageId": "c753e044-0d58-446c-aac5-b3ca4b4a520d",
          "contextId": "5bd6d5ec-31f4-48ad-aa1f-02e7f0ab6c3f",
          "taskId": "a4b57a5b-2340-43b4-a1fd-7e4b36327781",
          "role": "ROLE_AGENT",
          "parts": [
            {
              "text": "A-100 降噪耳機：NT$3990，有貨（庫存 42）"
            }
          ]
        },
        "timestamp": "2026-09-16T18:02:48.663102Z"
      },
      "artifacts": [
        {
          "artifactId": "e905b816-5ee1-4f44-9150-90ec5f326102",
          "name": "product",
          "parts": [
            {
              "data": {
                "price": 3990.0,
                "stock": 42.0,
                "sku": "A-100",
                "name": "降噪耳機"
              }
            }
          ]
        }
      ],
     

回應不是一句話，而是一個 **Task**。拆開來看：

In [7]:
task = resp["result"]["task"]
print("task id    :", task["id"], "  ← server 產生，追蹤這件工作")
print("context id :", task["contextId"], "  ← server 產生，串起多輪對話")
print("state      :", task["status"]["state"])
print("回覆文字   :", task["status"]["message"]["parts"][0]["text"])
print("artifacts  :", [a["name"] for a in task["artifacts"]])
print("history    :", [(m["role"], m["parts"][0]["text"]) for m in task["history"]])

product = task["artifacts"][0]["parts"][0]["data"]
print("\n結構化結果:", product)
print("⚠️ price 的型別:", type(product["price"]).__name__, "← server 送的是 int 3990")

task id    : a4b57a5b-2340-43b4-a1fd-7e4b36327781   ← server 產生，追蹤這件工作
context id : 5bd6d5ec-31f4-48ad-aa1f-02e7f0ab6c3f   ← server 產生，串起多輪對話
state      : TASK_STATE_COMPLETED
回覆文字   : A-100 降噪耳機：NT$3990，有貨（庫存 42）
artifacts  : ['product']
history    : [('ROLE_USER', 'A-100 多少錢？'), ('ROLE_AGENT', '收到，查詢目錄中…')]

結構化結果: {'price': 3990.0, 'stock': 42.0, 'sku': 'A-100', 'name': '降噪耳機'}
⚠️ price 的型別: float ← server 送的是 int 3990


兩個觀察：

1. **文字回覆和結構化結果是分開的。** 給人看的放 `status.message`，給程式用的放 `artifacts`。
   REST API 通常只有一個 body，你得自己決定兩者怎麼共存。
2. ⚠️ **`data` part 裡的整數變成了 float。** `data` 在協定裡是 protobuf `Struct`，
   而 `Struct` 的數字只有 `double` 一種。SKU 數量、金額這類欄位，**接收端要自己轉回 `int`**，
   或者乾脆用字串傳——這是跨框架整合時很容易漏的一個坑。

## 5. Task 生命週期：遠端 agent 回頭跟你要東西

這是 A2A 跟「開一個 REST endpoint」最不一樣的地方。先看協定定義了哪些狀態：

In [8]:
NOTE = {
    "TASK_STATE_UNSPECIFIED": "protobuf 的預設值，正常回應不會出現",
    "TASK_STATE_SUBMITTED": "已收件，還沒開始",
    "TASK_STATE_WORKING": "進行中（可以帶進度訊息）",
    "TASK_STATE_COMPLETED": "完成 ── 終止狀態",
    "TASK_STATE_FAILED": "失敗 ── 終止狀態",
    "TASK_STATE_CANCELED": "被取消 ── 終止狀態",
    "TASK_STATE_REJECTED": "拒絕受理 ── 終止狀態",
    "TASK_STATE_INPUT_REQUIRED": "⭐ 中斷：需要呼叫方補件",
    "TASK_STATE_AUTH_REQUIRED": "⭐ 中斷：需要呼叫方先完成驗證",
}
for s in T.TaskState.keys():
    print(f"  {s:28s} {NOTE.get(s, '')}")

  TASK_STATE_UNSPECIFIED       protobuf 的預設值，正常回應不會出現
  TASK_STATE_SUBMITTED         已收件，還沒開始
  TASK_STATE_WORKING           進行中（可以帶進度訊息）
  TASK_STATE_COMPLETED         完成 ── 終止狀態
  TASK_STATE_FAILED            失敗 ── 終止狀態
  TASK_STATE_CANCELED          被取消 ── 終止狀態
  TASK_STATE_INPUT_REQUIRED    ⭐ 中斷：需要呼叫方補件
  TASK_STATE_REJECTED          拒絕受理 ── 終止狀態
  TASK_STATE_AUTH_REQUIRED     ⭐ 中斷：需要呼叫方先完成驗證


現在故意**不講商品編號**，看 server 怎麼回。

In [9]:
r1 = rpc("SendMessage", user_msg("這個還有庫存嗎？"))["result"]["task"]
print("state:", r1["status"]["state"])
print("遠端說:", r1["status"]["message"]["parts"][0]["text"])

state: TASK_STATE_INPUT_REQUIRED
遠端說: 請提供商品編號（例如 A-100）


不是 `FAILED`，是 `INPUT_REQUIRED`。正確的做法是**帶著同一個 `taskId` 和 `contextId` 補件**：

In [10]:
r2 = rpc("SendMessage", user_msg("A-100", taskId=r1["id"], contextId=r1["contextId"]))["result"]["task"]
print("同一個 task？", r2["id"] == r1["id"])
print("state:", r2["status"]["state"])
print("遠端說:", r2["status"]["message"]["parts"][0]["text"])

print("\n=== 用 GetTask 把整段歷史撈回來（server 的 TaskStore 記著）===")
got = rpc("GetTask", {"id": r1["id"]})["result"]
for m in got["history"]:
    who = "👤" if m["role"] == "ROLE_USER" else "🤖"
    print(f"  {who} {m['parts'][0]['text']}")

同一個 task？ True
state: TASK_STATE_COMPLETED
遠端說: A-100 降噪耳機：NT$3990，有貨（庫存 42）

=== 用 GetTask 把整段歷史撈回來（server 的 TaskStore 記著）===
  👤 這個還有庫存嗎？
  🤖 收到，查詢目錄中…
  🤖 請提供商品編號（例如 A-100）
  👤 A-100
  🤖 收到，查詢目錄中…


**`taskId` 和 `contextId` 是兩件不同的事**，這是實作時最常搞混的：

| | 意義 | 只帶它會發生什麼 |
|---|---|---|
| `taskId` | **這一件工作** | 續接那個被中斷的 task（server 端 `current_task` 不是 `None`） |
| `contextId` | **這一段對話** | server 開一個**新的 task**，但知道你們在聊同一件事 |

驗證一下：只帶 `contextId`、不帶 `taskId`。

In [11]:
r3 = rpc("SendMessage", user_msg("那 C-300 呢？", contextId=r1["contextId"]))["result"]["task"]
print("新的 task？      ", r3["id"] != r1["id"])
print("同一段對話？     ", r3["contextId"] == r1["contextId"])
print("遠端說:", r3["status"]["message"]["parts"][0]["text"])

新的 task？       True
同一段對話？      True
遠端說: C-300 4K 螢幕：NT$11900，有貨（庫存 7）


換成 REST，「需要補件」你只能回 400 或自訂錯誤碼，每個團隊的約定都不一樣；
「接續同一件工作」得自己設計 session。**A2A 把這兩件事寫進了協定。**

> ⚠️ `AUTH_REQUIRED` 也一樣只是個**狀態值**。token 怎麼發、怎麼驗，協定不管——那是 Day 28 的事。

## 6. 串流：長任務邊做邊回報

卡片上寫了 `capabilities.streaming: true`，所以可以改用 `SendStreamingMessage`。
回應變成 **Server-Sent Events**：同一條 HTTP 連線上，server 每有一個事件就推一行 `data: {...}`。

「盤點」在這台 server 上是個約 5 秒的長任務。

In [12]:
def stream(text: str, **ids):
    body = {"jsonrpc": "2.0", "id": "s1", "method": "SendStreamingMessage", "params": user_msg(text, **ids)}
    with httpx.stream("POST", RPC_URL, json=body, headers=HEADERS, timeout=60) as r:
        print("Content-Type:", r.headers["content-type"], "\n")
        for line in r.iter_lines():
            if line.startswith("data:"):
                yield json.loads(line[5:])["result"]


t0 = time.monotonic()
for ev in stream("幫我盤點"):
    kind = next(iter(ev))  # task / statusUpdate / artifactUpdate / message
    body = ev[kind]
    state = body["status"]["state"].removeprefix("TASK_STATE_")
    msg = body["status"].get("message", {}).get("parts", [{}])[0].get("text", "")
    print(f"  +{time.monotonic() - t0:4.1f}s  {kind:<13} {state:<10} {msg}")

Content-Type: text/event-stream; charset=utf-8 

  + 0.0s  task          SUBMITTED  
  + 0.0s  statusUpdate  WORKING    收到，查詢目錄中…


  + 0.5s  statusUpdate  WORKING    盤點進度 10%


  + 1.0s  statusUpdate  WORKING    盤點進度 20%


  + 1.5s  statusUpdate  WORKING    盤點進度 30%


  + 2.0s  statusUpdate  WORKING    盤點進度 40%


...

  + 3.0s  statusUpdate  WORKING    盤點進度 60%


  + 3.5s  statusUpdate  WORKING    盤點進度 70%


  + 4.0s  statusUpdate  WORKING    盤點進度 80%


  + 4.5s  statusUpdate  WORKING    盤點進度 90%


  + 5.0s  statusUpdate  WORKING    盤點進度 100%
  + 5.0s  statusUpdate  COMPLETED  盤點完成，共 3 項商品


每一行都是一個**獨立的 JSON-RPC 回應**，外層的鍵告訴你事件種類：

| 鍵 | 什麼時候出現 |
|---|---|
| `task` | 第一個事件，Task 剛建立 |
| `statusUpdate` | 狀態或進度訊息改變（`TaskStatusUpdateEvent`） |
| `artifactUpdate` | 產出結果（`TaskArtifactUpdateEvent`，大檔案可以分段 `append`） |
| `message` | agent 選擇不開 Task、直接回一則訊息時 |

用 REST 做同樣的事，你得自己設計 polling 或 webhook。

## 7. 中途取消

長任務跑到一半不想等了？A2A 有 `CancelTask`。
先用 `returnImmediately: true` 送出（**不等做完就先回傳 Task**），再取消它。

In [13]:
params = user_msg("幫我盤點") | {"configuration": {"returnImmediately": True}}
started = rpc("SendMessage", params)["result"]["task"]
print("送出後立刻拿到:", started["status"]["state"])

time.sleep(1.2)
print("跑了 1.2 秒:  ", rpc("GetTask", {"id": started["id"]})["result"]["status"]["state"])

canceled = rpc("CancelTask", {"id": started["id"]})["result"]
print("CancelTask 後:", canceled["status"]["state"])

time.sleep(1.5)
final = rpc("GetTask", {"id": started["id"]})["result"]
last = [m["parts"][0]["text"] for m in final["history"] if m["role"] == "ROLE_AGENT"][-1]
print("再等 1.5 秒:  ", final["status"]["state"], f"（最後一則進度：{last}）")
print("\n→ 取消之後進度停住了：server 端的 execute() 真的被中斷，不是只改了個標籤。")

送出後立刻拿到: TASK_STATE_SUBMITTED


跑了 1.2 秒:   TASK_STATE_WORKING
CancelTask 後: TASK_STATE_CANCELED


再等 1.5 秒:   TASK_STATE_CANCELED （最後一則進度：盤點進度 20%）

→ 取消之後進度停住了：server 端的 execute() 真的被中斷，不是只改了個標籤。


取消一個**已經結束**的 task 會怎樣？終止狀態是不能再變的：

In [14]:
print(json.dumps(rpc("CancelTask", {"id": r2["id"]}), ensure_ascii=False, indent=2))

{
  "error": {
    "code": -32002,
    "message": "Task cannot be canceled",
    "data": [
      {
        "@type": "type.googleapis.com/google.rpc.ErrorInfo",
        "reason": "TASK_NOT_CANCELABLE",
        "domain": "a2a-protocol.org",
        "metadata": {}
      }
    ]
  },
  "id": "89b08b",
  "jsonrpc": "2.0"
}


## 8. ⚠️ 協定層的錯誤長什麼樣

A2A 沿用 JSON-RPC 的錯誤格式，並定義了自己的錯誤碼。先看**今天最容易踩的一個**：忘了帶 `A2A-Version` header。

In [15]:
print("=== 沒帶 A2A-Version header ===")
print(json.dumps(rpc("SendMessage", user_msg("A-100"), headers={}), ensure_ascii=False, indent=2))

=== 沒帶 A2A-Version header ===
{
  "error": {
    "code": -32009,
    "message": "A2A version '0.3' is not supported by this handler. Expected version '1.0'.",
    "data": [
      {
        "@type": "type.googleapis.com/google.rpc.ErrorInfo",
        "reason": "VERSION_NOT_SUPPORTED",
        "domain": "a2a-protocol.org",
        "metadata": {}
      }
    ]
  },
  "id": "713cb0",
  "jsonrpc": "2.0"
}


為什麼？`a2a-sdk` 1.x 同時支援舊版協定：**沒帶 header 就當成你講的是 0.3 版**，
而 1.0 的 `SendMessage` 方法不接受 0.3 的請求。這在「自己用 curl / Postman 測試」時幾乎一定會踩到。

（0.3 版的方法名是 `message/send`，1.0 改成 `SendMessage`——網路上的舊教學大多還是 0.3 的寫法。）

其他常見的錯誤：

In [16]:
cases = {
    "方法名打錯": ("SendMesage", user_msg("A-100")),
    "舊版方法名（1.0 server 未開相容模式）": ("message/send", user_msg("A-100")),
    "查不存在的 task": ("GetTask", {"id": "no-such-task"}),
    "參數欄位錯誤": ("SendMessage", {"msg": {}}),
}
for label, (method, params) in cases.items():
    err = rpc(method, params)["error"]
    print(f"{label:<28} code={err['code']:<7} {err['message'][:60]}")

方法名打錯                        code=-32601  Method not found
舊版方法名（1.0 server 未開相容模式）     code=-32601  Method not found
查不存在的 task                   code=-32001  Task not found
參數欄位錯誤                       code=-32602  Invalid params


| code | 意義 | 來源 |
|---|---|---|
| `-32601` | Method not found | JSON-RPC 標準 |
| `-32602` | Invalid params | JSON-RPC 標準 |
| `-32001` | Task not found | **A2A 定義** |
| `-32002` | Task not cancelable | **A2A 定義** |
| `-32009` | Version not supported | **A2A 定義** |

錯誤碼是協定的一部分，所以不管對方用什麼框架寫，你的錯誤處理都能寫一次就好。

## 9. Consuming：ADK 的 `RemoteA2aAgent` 替你做了什麼

前面手刻的每一件事——抓卡、選 URL、組 message、帶 header、記 `taskId` / `contextId`——
`RemoteA2aAgent` 都包掉了。它只要一個名字和卡片網址。

重點是：**A2A 的回應被翻譯成了 ADK Event**。我們寫一個小工具把 event 攤開來看。

In [17]:
from google.adk.agents.remote_a2a_agent import RemoteA2aAgent
from google.adk.runners import InMemoryRunner
from google.genai import types


def make_catalog():
    """每次給新實例——agent 只能有一個父節點（Day 16）。"""
    return RemoteA2aAgent(
        name="product_catalog",
        agent_card=CARD_URL,
        description="產品目錄服務：依 SKU 查詢品名、價格、庫存。",
    )


def show(ev):
    meta = ev.custom_metadata or {}
    tid = meta.get("a2a:task_id", "")[:8]
    for p in ev.content.parts if ev.content else []:
        if p.function_call:
            kind, val = "function_call", f"{p.function_call.name}({dict(p.function_call.args)})"
        elif p.function_response:
            kind, val = "function_resp", p.function_response.name
        elif p.inline_data:
            kind, val = "inline_data", p.inline_data.data.decode()[:70]
        elif p.thought:
            kind, val = "text (thought)", p.text
        else:
            kind, val = "text", p.text
        lr = " [long-running]" if ev.long_running_tool_ids else ""
        print(f"  {ev.author:<16} task={tid or '-':<9} {kind:<15} {val}{lr}")


async def send(runner, sid, message):
    events = []
    async for ev in runner.run_async(user_id="student", session_id=sid, new_message=message):
        show(ev)
        events.append(ev)
    return events


def text(s):
    return types.Content(role="user", parts=[types.Part(text=s)])


r_direct = InMemoryRunner(agent=make_catalog(), app_name="day19")
sid = await new_session(r_direct)
print("📥 B-200 還有貨嗎？")
evs = await send(r_direct, sid, text("B-200 還有貨嗎？"))

📥 B-200 還有貨嗎？
  product_catalog  task=5422f2df  text (thought)  收到，查詢目錄中…
  product_catalog  task=5422f2df  inline_data     <a2a_datapart_json>{"sku": "B-200", "price": 2490.0, "name": "\u6a5f\u
  product_catalog  task=5422f2df  text            B-200 機械鍵盤：NT$2490，缺貨（庫存 0）


同一個 A2A Task，在 ADK 這邊被拆成了幾種 Part。對照文章說的**三個核心能力**：

| A2A 那邊 | ADK 這邊變成 | 對應文章的能力 |
|---|---|---|
| `WORKING` 狀態附帶的訊息 | `text` 且 **`thought=True`** | **Reasoning**：過程訊息被當成推理軌跡保留，不混進最終答案 |
| `artifacts[].parts[].data` | `inline_data`，內容包在 `<a2a_datapart_json>` 裡 | **Artifacts** |
| `COMPLETED` 的 `status.message` | 一般 `text`，`is_final_response()` 為真 | — |
| `taskId` / `contextId` | `event.custom_metadata["a2a:task_id"]` 等 | 讓下一輪能續接 |

第三個能力 **Long-Running Tools** 要看 `INPUT_REQUIRED` 才會出現：

In [18]:
sid2 = await new_session(r_direct)
print("📥 這個還有貨嗎？（故意不給 SKU）")
evs = await send(r_direct, sid2, text("這個還有貨嗎？"))

pending = [fc for e in evs for fc in e.get_function_calls()][0]
first_task = evs[-1].custom_metadata["a2a:task_id"]
print("\n遠端的補件要求被包成:", pending.name)
print("它的 args:", dict(pending.args))
print("這個 event 的 long_running_tool_ids:", evs[-1].long_running_tool_ids)

📥 這個還有貨嗎？（故意不給 SKU）
  product_catalog  task=9b2b0f1f  text (thought)  收到，查詢目錄中…
  product_catalog  task=9b2b0f1f  function_call   mock_function_call_for_required_user_input({'input_required': '請提供商品編號（例如 A-100）'}) [long-running]

遠端的補件要求被包成: mock_function_call_for_required_user_input
它的 args: {'input_required': '請提供商品編號（例如 A-100）'}
這個 event 的 long_running_tool_ids: {'5f151727-c0bd-4175-bc62-ad13ed895157'}


ADK 沒有「遠端 agent 在等你補件」這個概念，所以它把 `INPUT_REQUIRED` **偽裝成一個 long-running 的 function call**。
這樣 Runner 就會停下來、把控制權交還給你——跟 Day 15 的 human-in-the-loop 是同一套機制。

⚠️ **要續接同一個 task，得用 `FunctionResponse` 回覆這個 call**，而不是直接再送一句話：

In [19]:
reply = types.Content(role="user", parts=[types.Part(function_response=types.FunctionResponse(
    id=pending.id, name=pending.name, response={"sku": "A-100"},
))])
print("📥 FunctionResponse(sku=A-100)")
evs = await send(r_direct, sid2, reply)
print("\n同一個 task？", evs[-1].custom_metadata["a2a:task_id"] == first_task)

📥 FunctionResponse(sku=A-100)
  product_catalog  task=9b2b0f1f  text (thought)  收到，查詢目錄中…
  product_catalog  task=9b2b0f1f  inline_data     <a2a_datapart_json>{"sku": "A-100", "price": 3990.0, "name": "\u964d\u
  product_catalog  task=9b2b0f1f  text            A-100 降噪耳機：NT$3990，有貨（庫存 42）

同一個 task？ True


In [20]:
sid3 = await new_session(r_direct)
evs = await send(r_direct, sid3, text("這個還有貨嗎？"))
first = evs[-1].custom_metadata["a2a:task_id"]
print("\n📥 改成直接送文字 'A-100'")
evs = await send(r_direct, sid3, text("A-100"))
print("\n同一個 task？", evs[-1].custom_metadata["a2a:task_id"] == first,
      "← 只帶了 contextId，server 開了新 task")

  product_catalog  task=2610be11  text (thought)  收到，查詢目錄中…
  product_catalog  task=2610be11  function_call   mock_function_call_for_required_user_input({'input_required': '請提供商品編號（例如 A-100）'}) [long-running]

📥 改成直接送文字 'A-100'


  product_catalog  task=b457d96a  text (thought)  收到，查詢目錄中…
  product_catalog  task=b457d96a  inline_data     <a2a_datapart_json>{"sku": "A-100", "price": 3990.0, "name": "\u964d\u
  product_catalog  task=b457d96a  text            A-100 降噪耳機：NT$3990，有貨（庫存 42）

同一個 task？ False ← 只帶了 contextId，server 開了新 task


兩種寫法**結果都答對了**，差別在協定層：

- `FunctionResponse` → ADK 帶上原本的 `taskId`，**續接被中斷的那件工作**
- 直接送文字 → ADK 只帶 `contextId`，**開了一件新工作**，舊的那件永遠停在 `INPUT_REQUIRED`

這台 server 很單純，所以看不出差異。但如果對方的 task 在中斷前已經做了一半（例如已經鎖了庫存），
第二種寫法會讓那一半**永遠懸在那裡**。

## 10. 把遠端 agent 放進 LLM 團隊

`RemoteA2aAgent` 是一個 `BaseAgent`，所以可以直接放進 `sub_agents`。
對 coordinator 來說，**它跟本地 agent 長得一模一樣**——文章說的「跟呼叫本地工具幾乎沒有分別」就是這個意思。

In [21]:
from google.adk.agents import LlmAgent

customer_service = LlmAgent(
    name="customer_service",
    model=get_model(),
    instruction=(
        "你是電商客服。產品價格、庫存的問題一律轉給 product_catalog。"
        "其他問題一律回覆「這個問題我幫您轉給真人客服」，不要自己編答案。"
    ),
    sub_agents=[make_catalog()],
)

r_cs = InMemoryRunner(agent=customer_service, app_name="day19")
for q in ["C-300 的 4K 螢幕多少錢？", "我想退貨，要怎麼辦？"]:
    sid_cs = await new_session(r_cs)
    print(f"📥 {q}")
    await ask(r_cs, q, session_id=sid_cs, trace=True)
    print()

📥 C-300 的 4K 螢幕多少錢？


  🔧 [customer_service] 呼叫 transfer_to_agent({'agent_name': 'product_catalog'})
  ↩️  [customer_service] transfer_to_agent 回傳 {'result': None}
  💭 [product_catalog] （推理中，已隱藏）
  💬 [product_catalog] C-300 4K 螢幕：NT$11900，有貨（庫存 7）

📥 我想退貨，要怎麼辦？


  💬 [customer_service] 這個問題我幫您轉給真人客服



第一題走了 `transfer_to_agent` → HTTP → 另一個程序；第二題 coordinator 自己處理，**完全沒有網路流量**。
LLM 是靠卡片上的 `description` 決定要不要轉的——**卡片寫得爛，遠端 agent 就不會被呼叫**。

## 11. 兩條路同時走：consuming 的 agent 再 exposing 出去

文章說「大多數真實系統兩者都會用到」。`customer_service_server.py` 就是這樣：
它用 `RemoteA2aAgent` 接產品目錄，自己再用 `to_a2a()` 開出去。現在呼叫鏈有**三個程序**。

In [22]:
print(inspect.getsource(__import__("customer_service_server").build_agent))

CS = start_server(
    "customer_service", "customer_service_server.py", CS_PORT,
    "--catalog-port", str(CATALOG_PORT),
)

def build_agent(catalog_port: int) -> LlmAgent:
    product_catalog = RemoteA2aAgent(
        name="product_catalog",
        # ⚠️ 必須跟對方卡片上宣告的 host 完全一致（localhost ≠ 127.0.0.1，見 Day 20）
        agent_card=f"http://localhost:{catalog_port}/.well-known/agent-card.json",
        description="產品目錄服務：依 SKU 查詢品名、價格、庫存。",
    )
    return LlmAgent(
        name="customer_service_agent",
        model=get_model(),
        description="電商客服：回答產品價格與庫存問題。",
        instruction=(
            "你是電商客服。凡是產品價格、庫存的問題，一律轉給 product_catalog，"
            "拿到結果後用繁體中文、一到兩句話回覆顧客。"
        ),
        sub_agents=[product_catalog],
    )



✅ customer_service 已在 http://localhost:8942 上線（pid=389572）


這一次卡片**不是手寫的**，是 `to_a2a()` 從 agent 自動產生的。對照一下兩張卡：

In [23]:
cs_card = httpx.get(f"{CS}/.well-known/agent-card.json").json()
print(json.dumps(cs_card, ensure_ascii=False, indent=2))

{
  "name": "customer_service_agent",
  "description": "電商客服：回答產品價格與庫存問題。",
  "supportedInterfaces": [
    {
      "url": "http://localhost:8942",
      "protocolBinding": "JSONRPC",
      "protocolVersion": "1.0"
    }
  ],
  "version": "0.0.1",
  "capabilities": {
    "streaming": false,
    "pushNotifications": false
  },
  "defaultInputModes": [
    "text/plain"
  ],
  "defaultOutputModes": [
    "text/plain"
  ],
  "skills": [
    {
      "id": "customer_service_agent",
      "name": "model",
      "description": "電商客服：回答產品價格與庫存問題。",
      "tags": [
        "llm"
      ]
    },
    {
      "id": "product_catalog_product_catalog",
      "name": "product_catalog: custom",
      "description": "產品目錄服務：依 SKU 查詢品名、價格、庫存。",
      "tags": [
        "sub_agent:product_catalog",
        "custom_agent"
      ]
    }
  ]
}


| | product_catalog（手寫） | customer_service（`to_a2a` 自動產生） |
|---|---|---|
| `skills` 來源 | 自己一條條寫 | agent 本體一條 + **每個 sub_agent 各一條** |
| `description` | 自己寫 | `LlmAgent(description=...)` |
| `capabilities.streaming` | 自己宣告 | ADK 決定 |

⚠️ 注意 `skills` 裡出現了 `product_catalog`——**你的內部架構被寫進了公開的卡片**。
Day 20 會看到工具的 docstring 也一樣會被公開。

現在用第 4 節的同一個 `rpc()` 手刻請求，打這台 ADK 做的 server。**client 端不知道對方是不是 ADK。**

In [24]:
cs_rpc_url = cs_card["supportedInterfaces"][0]["url"]
t0 = time.monotonic()
cs_resp = httpx.post(
    cs_rpc_url, headers=HEADERS, timeout=120,
    json={"jsonrpc": "2.0", "id": "cs1", "method": "SendMessage",
          "params": user_msg("B-200 機械鍵盤還有貨嗎？")},
).json()
elapsed = time.monotonic() - t0

cs_task = cs_resp["result"]["task"]
print(f"state: {cs_task['status']['state']}（{elapsed:.1f}s）")
print("\n最終回覆:")
for a in cs_task.get("artifacts", []):
    for p in a["parts"]:
        if "text" in p:
            print("  ", p["text"])

state: TASK_STATE_COMPLETED（2.8s）

最終回覆:
   B-200 機械鍵盤：NT$2490，缺貨（庫存 0）


這一趟實際走了：`notebook ─HTTP→ customer_service (ADK, LLM) ─HTTP→ product_catalog (a2a-sdk)`。
看 customer_service 的 task 歷史，能看到它對下游發了什麼：

In [25]:
for m in cs_task.get("history", []):
    for p in m["parts"]:
        content = p.get("text") or json.dumps(p.get("data", {}), ensure_ascii=False)
        print(f"  {m['role']:<11} {content[:110]}")

  ROLE_USER   B-200 機械鍵盤還有貨嗎？
  ROLE_AGENT  {"name": "transfer_to_agent", "id": "call_30225", "args": {"agent_name": "product_catalog"}}
  ROLE_AGENT  {"name": "transfer_to_agent", "id": "call_30225", "response": {"result": null}}
  ROLE_AGENT  收到，查詢目錄中…
  ROLE_AGENT  {"name": "機械鍵盤", "sku": "B-200", "stock": 0.0, "price": 2490.0}
  ROLE_AGENT  B-200 機械鍵盤：NT$2490，缺貨（庫存 0）


⚠️ **外部呼叫方拿到了你的內部細節**：`transfer_to_agent` 的呼叫、sub_agent 的名稱、
下游服務回傳的原始資料，全部出現在回給外部的 task history 裡。
如果下游資料含有不該外流的欄位（成本價、內部代碼），**它們會一路穿透到最外層的 client**。
`historyLength` 參數救不了你——那是**呼叫方**決定要不要帶的。
要對外開放的 agent，得在 server 端自己過濾（例如自訂 `agent_executor_factory`，或讓對外的 agent 只拿到整理過的結果）。

## 12. 代價：同一段邏輯，本地呼叫 vs A2A

最後回到文章的第一個問題。產品目錄的查詢邏輯就是 `pcs.lookup()` 這個純函式，
我們直接比「在程序內呼叫它」和「隔著 A2A 呼叫它」。**這一段不含 LLM，純粹是協定的成本。**

In [26]:
import statistics
import timeit

N = 200
local_s = timeit.timeit(lambda: pcs.lookup("A-100"), number=N) / N

with httpx.Client(headers=HEADERS) as client:
    client.post(RPC_URL, json={"jsonrpc": "2.0", "id": 0, "method": "SendMessage", "params": user_msg("A-100")})
    samples = []
    for _ in range(N):
        t0 = time.perf_counter()
        client.post(RPC_URL, json={"jsonrpc": "2.0", "id": 0, "method": "SendMessage", "params": user_msg("A-100")})
        samples.append(time.perf_counter() - t0)

a2a_s = statistics.median(samples)
print(f"本地函式呼叫  : {local_s * 1e6:8.1f} µs")
print(f"A2A（同一台機器、keep-alive）: {a2a_s * 1e6:8.1f} µs   ≈ {a2a_s / local_s:,.0f} 倍")
print(f"A2A p95       : {sorted(samples)[int(N * 0.95)] * 1e6:8.1f} µs")
body = json.dumps(rpc("SendMessage", user_msg("A-100")), ensure_ascii=False)
print(f"\n本地回傳一個 dict；A2A 回傳 {len(body.encode())} bytes 的 JSON（含完整 history）")

本地函式呼叫  :      0.4 µs
A2A（同一台機器、keep-alive）:   1269.7 µs   ≈ 3,619 倍
A2A p95       :   2041.2 µs

本地回傳一個 dict；A2A 回傳 1147 bytes 的 JSON（含完整 history）


這還是**同一台機器、沒有 TLS、沒有驗證**的最好情況。跨機房再加上幾十毫秒，
還多了一個會在半夜掛掉的服務要顧。

所以文章裡把 `DataValidator` 拆成 A2A 服務的團隊，換到的是：

- 一次呼叫慢上數百到數千倍
- 每次都要序列化 / 反序列化（而且整數還會變 float）
- 失去共享 session state 的能力
- 多一個要部署、監控、升級的東西

**判斷準則：如果你說不出「對方是獨立的服務 / 別的團隊 / 別的語言 / 需要正式契約」其中一個，就不要用 A2A。**

## 13. 📌 補充：A2A Extension

文章提到 ADK 另有一份 [A2A Extension](https://adk.dev/a2a/a2a-extension/)，用改版過的 `A2aAgentExecutor`
修掉 **A2A 與 ADK 同時開串流**時的三個問題：

1. 使用者訊息在 task history 裡重複
2. 遠端 agent 的輸出被誤判成 thought
3. 巢狀 sub-agent 的輸出遺失

第 2 點你在第 9 節已經親眼看過它的「正常版本」：`WORKING` 狀態的訊息被標成 `thought=True`。
如果你發現**遠端 agent 的最終答案**也被標成 thought、`ask()` 回傳空字串，就是該回頭查這份文件的時候。

## 14. 收工：關掉子程序

⚠️ 忘了關，埠會一直被佔著，下次重跑會在第 2 節報「埠已被佔用」。

In [27]:
stop_all()
for name, proc in PROCS.items():
    print(f"{name:<18} 回傳碼={proc.returncode}")
print("8941 還開著？", port_in_use(CATALOG_PORT))
print("8942 還開著？", port_in_use(CS_PORT))
shutil.rmtree(WORK, ignore_errors=True)

product_catalog    回傳碼=-15
customer_service   回傳碼=-15
8941 還開著？ False
8942 還開著？ False


## 15. 常見錯誤與踩坑

| 症狀 | 原因 |
|---|---|
| `No module named 'a2a'` | 沒裝 `google-adk[a2a]`，或 **notebook 選錯 kernel** |
| `code=-32009 A2A version '0.3' is not supported` | 手刻請求**沒帶 `A2A-Version: 1.0` header** |
| `code=-32601 Method not found` | 用了 0.3 的 `message/send`；1.0 是 `SendMessage` |
| 補件後遠端當成新問題處理 | 只帶了 `contextId`，**沒帶 `taskId`**；ADK 端要用 `FunctionResponse` 回覆 |
| 收到的數量變成 `42.0` | `data` part 是 protobuf `Struct`，數字一律 double |
| ADK 端看到 `mock_function_call_for_required_user_input` | 遠端進入 `INPUT_REQUIRED`，不是 bug |
| 遠端的進度訊息沒出現在 `ask()` 結果 | `WORKING` 訊息被轉成 `thought=True`，`ask()` 會濾掉 |
| 卡片上出現內部 sub_agent 名稱 | `to_a2a()` 自動產生卡片時會列出 sub_agents |
| 外部 client 看到內部的 `transfer_to_agent` 與下游原始資料 | `to_a2a()` 回傳的 task history 包含整條內部呼叫鏈 |
| 同程序的 agent 也用 A2A 串 | 過度設計，第 12 節的延遲就是代價 |
| 第 2 節報「埠已被佔用」 | 上次沒跑到第 14 節，先關掉舊程序 |

## 16. 動手練習

1. **補一個狀態**：在 `product_catalog_server.py` 裡，當訊息含「進貨價」時改走 `updater.requires_auth()`。
   用 httpx 打一次，再用 `RemoteA2aAgent` 打一次，看 ADK 把它翻成哪個 mock function。
2. **修掉 float 坑**：把 artifact 改成 `new_text_part(json.dumps(product))` 並設 `media_type="application/json"`，
   確認接收端拿到的 `stock` 是 int。
3. **看卡片的影響**：把 `product_catalog` 的 `description` 改成「查天氣」，重跑第 10 節，
   觀察 coordinator 還會不會把價格問題轉過去。
4. **換一個呼叫方**：用 `curl` 在終端機打 `SendMessage`（記得帶 header），證明 client 不需要 Python。
5. **取消 × 串流**：一邊用第 6 節的 `stream()` 跑盤點，另一個 cell 對它 `CancelTask`，看串流最後收到什麼。

## 本日回顧

- **A2A 解決的是「跨程序、跨團隊、跨框架」**；同程序就用 sub-agent，第 12 節的延遲就是理由。
- **Agent Card** 是機器可讀的契約：`supportedInterfaces` 告訴你往哪打，`skills` 是給 LLM 的路由表。
- 線上格式是 **JSON-RPC 2.0**，方法名 `SendMessage` / `SendStreamingMessage` / `GetTask` / `CancelTask`；
  ⚠️ 手刻時要帶 **`A2A-Version: 1.0`**。
- **Task 生命週期**是 A2A 勝過 REST 的地方：`INPUT_REQUIRED` 補件、SSE 進度、`CancelTask` 真的會中斷執行。
- ⚠️ **`taskId` 續接工作，`contextId` 續接對話**；ADK 端要用 `FunctionResponse` 回覆才會帶上 `taskId`。
- `RemoteA2aAgent` 把 A2A 翻成 ADK Event：進度 → thought、artifact → inline_data、補件 → long-running function call。
- 一支 agent 可以**同時 consuming 與 exposing**；⚠️ 但 `to_a2a()` 的卡片與 task history 都會洩漏內部結構。

---
**下一天 → `../day20_a2a_exposing_consuming/`**：專注在 `to_a2a()` 的細節、agent card 欄位來源，以及 `localhost` vs `127.0.0.1` 的靜默失敗。